<a href="https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-2/10-miniproyecto-reporte-de-la-manana/clase.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Episodio 2.10 · Miniproyecto: el reporte de la mañana

**Serie:** Del Excel al Machine Learning · **Temporada 2** · 🎬 Video próximamente

## 🎯 Qué vas a aprender

- Reunir pronóstico, árbol y texto en un **solo reporte**.
- Usar **solo lo que existe esa mañana**: nada de datos del futuro.
- **Comprobar** todo con lo que pasó después.

## 📊 En Excel lo harías así

Cada mañana abres tres archivos, buscas los pedidos de hoy, revisas la bandeja de mensajes, y armas un resumen para el equipo.

Hoy lo armamos con todo lo que aprendimos en la temporada. Es diciembre de 2025, el mes más movido.

## 🐍 En Python

### Paso 1: las piezas

Traemos las funciones de los episodios anteriores. Ya las conoces, así que basta con ejecutar estas celdas.

> ⚠️ **Datos sintéticos.** Todo esto es un ejercicio de armado, no una receta para tu empresa.

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier

DATOS = "https://raw.githubusercontent.com/PercyTechX/del-excel-al-ml/main/datos/"

In [2]:
FERIADOS = pd.to_datetime(
    [
        "2024-01-01",
        "2024-03-28",
        "2024-03-29",
        "2024-05-01",
        "2024-06-29",
        "2024-07-28",
        "2024-07-29",
        "2024-08-30",
        "2024-10-08",
        "2024-11-01",
        "2024-12-08",
        "2024-12-09",
        "2024-12-25",
        "2025-01-01",
        "2025-04-17",
        "2025-04-18",
        "2025-05-01",
        "2025-06-29",
        "2025-07-28",
        "2025-07-29",
        "2025-08-30",
        "2025-10-08",
        "2025-11-01",
        "2025-12-08",
        "2025-12-09",
        "2025-12-25",
    ]
)

In [3]:
def calendario(dias):
    """Una columna por día de la semana y por mes, más la tendencia: los días desde el inicio."""
    x = pd.DataFrame(index=dias)
    for dia in range(7):
        x[f"dia_{dia}"] = (dias.dayofweek == dia).astype(int)
    for mes in range(1, 13):
        x[f"mes_{mes}"] = (dias.month == mes).astype(int)
    x["tendencia"] = (dias - pd.Timestamp("2024-01-01")).days
    return x

In [4]:
def tres_pronosticos(serie, feriados, mes):
    """Pronostica cada día de un mes con tres métodos, usando solo lo anterior a ese mes."""
    inicio = pd.Timestamp(mes + "-01")
    dias = pd.date_range(inicio, inicio + pd.offsets.MonthEnd(0))
    antes = serie[serie.index < inicio]
    laborables = antes[~antes.index.isin(feriados)]

    # Promedio por día de la semana de las últimas 8 semanas.
    ultimas = laborables[laborables.index >= inicio - pd.Timedelta(days=56)]
    por_dia = ultimas.groupby(ultimas.index.dayofweek).mean()
    ocho_semanas = pd.Series(dias.dayofweek.map(por_dia), index=dias)

    # Índice: cuánto subió ese mes en 2024 frente a las 8 semanas previas de 2024.
    hace_un_anio = inicio - pd.Timedelta(days=364)
    base = laborables[
        (laborables.index >= hace_un_anio - pd.Timedelta(days=56))
        & (laborables.index < hace_un_anio)
    ]
    mismo_mes = laborables[
        (laborables.index.year == 2024) & (laborables.index.month == inicio.month)
    ]
    regla = ocho_semanas * mismo_mes.mean() / base.mean()

    modelo = LinearRegression().fit(calendario(laborables.index), laborables)
    regresion = pd.Series(modelo.predict(calendario(dias)), index=dias)

    # Todos los métodos ponen los feriados en cero: esos días no se vende.
    tabla = pd.DataFrame(
        {"real": serie[dias], "ocho_semanas": ocho_semanas, "regla": regla, "regresion": regresion}
    )
    metodos = ["ocho_semanas", "regla", "regresion"]
    tabla.loc[tabla.index.isin(feriados), metodos] = 0
    return tabla

In [5]:
def preparar_despachos(despachos):
    """Deja los datos que se conocen al preparar un despacho: textos como 0 y 1, mes y día."""
    fecha = pd.to_datetime(despachos["fecha"])
    x = pd.get_dummies(despachos[["ciudad", "tipo", "transportista"]], dtype=int)
    x = x.drop(columns=["ciudad_Lima", "tipo_Bodega", "transportista_Propio"], errors="ignore")
    x["mes"] = fecha.dt.month
    x["dia"] = fecha.dt.dayofweek
    return x.join(despachos[["distancia_km", "lineas", "presentaciones"]])

In [6]:
ventas = pd.read_csv(DATOS + "ventas.csv", parse_dates=["fecha"])
despachos = pd.read_csv(DATOS + "despachos.csv")
tickets = pd.read_csv(DATOS + "tickets.csv")

serie = ventas.groupby("fecha")["total"].sum().asfreq("D", fill_value=0)

### Paso 2: entrenar una sola vez

"Hoy" es el cierre del 30 de noviembre de 2025. Entrenamos con todo lo anterior a diciembre: el árbol del 2.2, el lector de tickets del 2.6, y el pronóstico del 2.4.

Sin esta línea entre pasado y futuro, todo lo demás perdería sentido.

In [7]:
previos = despachos[despachos["fecha"] < "2025-12-01"]
arbol = DecisionTreeClassifier(max_depth=6, random_state=0)
arbol.fit(preparar_despachos(previos), previos["llego_tarde"])

mensajes = tickets[tickets["fecha"] < "2025-12-01"]
conteo = CountVectorizer()
lector = LogisticRegression(max_iter=2000)
lector.fit(conteo.fit_transform(mensajes["mensaje"]), mensajes["categoria"])

pronostico = tres_pronosticos(serie, FERIADOS, "2025-12")

### Paso 3: el reporte de una mañana

Cada mañana de diciembre solo usamos lo que ya existe: **los pedidos tomados para hoy** y **los mensajes de ayer**. Vigilamos el 10 % de los despachos de cada día. Suponemos que los pedidos de hoy se cerraron antes del reporte: si entran durante el día, la lista se queda corta.

In [8]:
def reporte_de_la_manana(dia):
    """Lo que se sabe esa mañana: venta esperada, despachos a vigilar y tickets de ayer por área."""
    ayer = str((pd.Timestamp(dia) - pd.Timedelta(days=1)).date())
    hoy = despachos[despachos["fecha"] == dia].sample(frac=1, random_state=0)
    columnas = ["id_despacho", "id_cliente", "ciudad", "transportista", "riesgo"]
    if hoy.empty:  # Feriado: no hay despachos.
        vigilar = pd.DataFrame(columns=columnas)
    else:
        x = preparar_despachos(hoy).reindex(columns=arbol.feature_names_in_, fill_value=0)
        riesgo = arbol.predict_proba(x)[:, 1].round(2)
        cupo = max(1, round(len(hoy) * 0.10))
        vigilar = hoy.assign(riesgo=riesgo).nlargest(cupo, "riesgo")[columnas]

    de_ayer = tickets[tickets["fecha"] == ayer]
    por_area = de_ayer.assign(area="")[["id_ticket", "area", "mensaje"]]
    if not de_ayer.empty:
        por_area["area"] = lector.predict(conteo.transform(de_ayer["mensaje"]))

    venta = pronostico.loc[dia, ["regla", "regresion"]].round(0)
    return venta, vigilar, por_area

In [9]:
venta, vigilar, por_area = reporte_de_la_manana("2025-12-15")

print(venta)
print(por_area["area"].value_counts())

vigilar.head()

regla        34903.0
regresion    31480.0
Name: 2025-12-15 00:00:00, dtype: float64
area
Despacho              4
Producto              1
Consulta comercial    1
Name: count, dtype: int64


,id_despacho,id_cliente,ciudad,transportista,riesgo
40570,D40571,C0024,Cusco,Propio,0.93
40620,D40621,C0199,Trujillo,Propio,0.88
40596,D40597,C0128,Cusco,Tercero,0.57
40639,D40640,C0288,Cusco,Tercero,0.57
40585,D40586,C0078,Cusco,Propio,0.57


Y lo guardamos en Excel, con una hoja de notas, como en el 1.9.

In [10]:
notas = pd.DataFrame(
    {
        "nota": [
            "Reporte del 15 de diciembre de 2025, con lo que se sabía esa mañana.",
            "La lista de despachos es un orden de prioridad, no una predicción segura.",
            "Una persona revisa el área de cada mensaje antes de enviarlo.",
            "Datos sintéticos.",
        ]
    }
)

with pd.ExcelWriter("reporte_2025_12_15.xlsx", engine="openpyxl") as archivo:
    notas.to_excel(archivo, sheet_name="Notas", index=False)
    venta.reset_index().to_excel(archivo, sheet_name="Venta esperada", index=False)
    vigilar.to_excel(archivo, sheet_name="Despachos", index=False)
    por_area.to_excel(archivo, sheet_name="Tickets", index=False)

Búscalo en la carpeta 📁 de Colab y descárgalo, como en el 0.8.

### Paso 4: ¿funcionó?

Ahora vamos un paso más allá de lo que podría ver el equipo: dejamos que "hoy" avance cada día de diciembre, y comparamos con lo que pasó.

In [11]:
print((pronostico["real"] - pronostico["regla"]).abs().mean().round(0))
print((pronostico["real"] - pronostico["regresion"]).abs().mean().round(0))
print(pronostico["real"].mean().round(0))

3524.0
3486.0
25630.0


In [12]:
encontrados = 0
for dia in pd.date_range("2025-12-01", "2025-12-31").strftime("%Y-%m-%d"):
    venta, vigilar, por_area = reporte_de_la_manana(dia)
    marcados = despachos[despachos["id_despacho"].isin(vigilar["id_despacho"])]
    encontrados += marcados["llego_tarde"].sum()

diciembre = despachos[despachos["fecha"] >= "2025-12-01"]
print(len(diciembre), diciembre["llego_tarde"].sum())
print(round(encontrados / diciembre["llego_tarde"].sum(), 3))

2187 294
0.395


In [13]:
de_diciembre = tickets[tickets["fecha"] >= "2025-12-01"]

print(len(de_diciembre))
print(round(lector.score(conteo.transform(de_diciembre["mensaje"]), de_diciembre["categoria"]), 3))

187
0.909


Y así se leen:

- **Venta:** la regla y la regresión quedan a la par, con un error cercano al 14 % de la venta media por día. Con otros datos, gana a veces una y a veces la otra.
- **Despachos:** vigilando el 10 % se encuentra cerca del 40 % de los tardíos. Es mucho más que el 10 % al azar, y más que el 29 % del semestre, porque diciembre trae campaña y lluvia. No lo traslades al año. Y 6 de cada 10 tardíos no se ven venir.
- **Tickets:** cerca del 91 %, en solo 187 mensajes. Este lector se entrenó hasta noviembre, así que ya conoce las frases que empezaron en julio. Son solo 10 y se repiten, así que las memoriza: con mensajes reales, reentrenar ayudaría menos. En el ejercicio verás qué pasa con uno que no se reentrenó.

Es **un solo mes**, y ya lo habíamos visto al elegir estos métodos. Sirve para comprobar que el proceso funciona, no para declarar qué modelo es mejor.

### El reporte no reemplaza a nadie

Prioriza qué mirar primero. Las decisiones siguen siendo del equipo: llamar a un transportista, avisar a un cliente, corregir el área de un mensaje.

## 🤖 Pídeselo a la IA

> *"Tengo un reporte diario con venta esperada, una lista de despachos a vigilar y mensajes por área. ¿Qué cinco cosas debería revisar cada mes para saber si sigue funcionando?"*

La IA te ayuda a escribir y entender código, pero **tú debes entender lo que hace**. Revisa cada respuesta antes de usarla, y no pegues datos de tu empresa.

## ✅ Resumen

| En Excel | En Python |
|---|---|
| Abrir tres archivos cada mañana | `reporte_de_la_manana(dia)` |
| Solo lo que ya llegó | `despachos["fecha"] == dia` y mensajes de ayer |
| Guardar el reporte | `pd.ExcelWriter()` y `.to_excel()` |
| Comprobar con lo que pasó | Error y parte encontrada de todo diciembre |

## 🏁 Terminaste la Temporada 2

Ya sabes usar árboles, bosques, agrupar clientes, pronosticar por día y clasificar mensajes. Y lo más importante: sabes cuándo un modelo no aporta, y cuándo no se debe usar. En la **Temporada 3** haremos un proyecto completo con datos reales, hasta dejarlo funcionando.

## 📝 Ahora te toca

Abre [el ejercicio en Colab](https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-2/10-miniproyecto-reporte-de-la-manana/ejercicio.ipynb) y arma el reporte de otra mañana. Cuando termines, compara con [la solución](https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-2/10-miniproyecto-reporte-de-la-manana/solucion.ipynb).

---

*Del Excel al Machine Learning* · por [Percy Nazario](https://www.linkedin.com/in/percy-nazario/) · [Repositorio](https://github.com/PercyTechX/del-excel-al-ml) · Contenido bajo licencia CC BY 4.0